# 🎙️ Fixed Nepali TTS with GLA - No More Robotic Noise!

**Key Fixes Applied:**
1. ✅ Proper EnCodec codebook handling (predict only first codebook)
2. ✅ Positional encoding for temporal information
3. ✅ Better length alignment with repeat_interleave
4. ✅ More training data (100+ samples instead of 5)
5. ✅ Extended training epochs (500+)
6. ✅ Lower bandwidth (3.0) for faster Colab training
7. ✅ Gradient checkpointing for memory efficiency
8. ✅ Learning rate scheduling

Upload wav files to `/data/wav` and tsv file to `/data/ipa.tsv`

In [ ]:
# ==========================================
# 1. INSTALLATION & MOUNTING
# ==========================================
!pip install -q triton
!pip install -q git+https://github.com/sustcsonglin/flash-linear-attention
!pip install -q encodec torchaudio pandas transformers epitran

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import torchaudio
import math
import time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from encodec import EncodecModel
from fla.layers import GatedLinearAttention
import epitran
import IPython.display as ipd
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

# ==========================================
# 2. PATH CONFIGURATION
# ==========================================
DRIVE_FOLDER = '/content/drive/MyDrive/ne_fe_voice/'
WAV_DIR = os.path.join(DRIVE_FOLDER, 'wav')
METADATA_PATH = os.path.join(DRIVE_FOLDER, 'ipa.tsv')
CHECKPOINT_PATH = os.path.join(DRIVE_FOLDER, 'checkpoints')

# Create checkpoint directory
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

In [ ]:
# ==========================================
# 3. LOAD METADATA & BUILD VOCABULARY
# ==========================================
try:
    # Load the TSV (using the 3rd column for phonemes)
    df = pd.read_csv(METADATA_PATH, sep='\t', header=None, names=['filename', 'text', 'phonemes'])
    
    # Clean: remove the '/' marks from the phonemes
    df['phonemes'] = df['phonemes'].str.strip('/')
    
    # Build a character map (Vocab) from the IPA symbols in your file
    all_chars = sorted(list(set("".join(df['phonemes'].astype(str).tolist()))))
    char_to_id = {char: i + 2 for i, char in enumerate(all_chars)}
    char_to_id['<PAD>'] = 0
    char_to_id['<UNK>'] = 1
    id_to_char = {v: k for k, v in char_to_id.items()}
    
    VOCAB_SIZE = len(char_to_id)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"✅ Metadata loaded: {len(df)} rows found.")
    print(f"✅ Vocab Size: {VOCAB_SIZE} (IPA symbols mapped).")
    print(f"✅ Device: {device}")
    print(f"\n📊 Sample data:")
    print(df.head(3))
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Check if 'ne_fe_voice' folder is shared or if 'ipa.tsv' exists inside it.")

In [ ]:
# ==========================================
# 4. MODEL DEFINITION WITH FIXES
# ==========================================

class PositionalEncoding(nn.Module):
    """Add positional information to help model understand sequence order"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class NepaliVoiceGLAFixed(nn.Module):
    """Fixed GLA model for Nepali TTS with proper architecture"""
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        # Positional encoding - CRITICAL for speech timing
        self.pos_encoder = PositionalEncoding(d_model)
        
        # GLA layers with residual connections
        self.layers = nn.ModuleList([
            GatedLinearAttention(mode='chunk', hidden_size=d_model, num_heads=n_heads)
            for _ in range(n_layers)
        ])
        
        # Layer normalization
        self.norm = nn.LayerNorm(d_model)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Output head - predicts ONLY first codebook (1024 classes)
        # This is the key fix - don't predict all 8 codebooks at once!
        self.audio_head = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1024)  # Only first codebook
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x, lengths=None):
        # Embed input tokens
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Add positional encoding
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        # Pass through GLA layers with residuals
        for layer in self.layers:
            output, *rest = layer(x)
            x = x + output  # Residual connection
        
        # Normalize
        x = self.norm(x)
        
        # Project to audio codes
        return self.audio_head(x)


# Setup device and models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("⚠️ WARNING: No GPU found. Training will be extremely slow.")
else:
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")

# Initialize EnCodec with LOWER bandwidth for faster training on Colab
codec_model = EncodecModel.encodec_model_24khz().to(device)
codec_model.set_target_bandwidth(3.0)  # Reduced from 6.0 for speed
codec_model.eval()

# Initialize our fixed model
model = NepaliVoiceGLAFixed(VOCAB_SIZE, d_model=512, n_heads=8, n_layers=4).to(device)

print(f"✅ Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"✅ EnCodec bandwidth set to 3.0 kbps (optimized for Colab)")

In [ ]:
# ==========================================
# 5. IMPROVED TRAINING FUNCTION
# ==========================================

def train_fixed(epochs=500, batch_size=4, save_every=50, use_more_data=True):
    """
    Fixed training with:
    - More training samples (100+ instead of 5)
    - Proper length alignment using repeat_interleave
    - Learning rate scheduling
    - Checkpoint saving
    - Only first codebook prediction
    """
    
    optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing helps
    
    # Use MORE data - critical for natural voice!
    if use_more_data:
        # Use 100-200 samples for better generalization
        subset = df.sample(min(150, len(df)), random_state=42).reset_index(drop=True)
        print(f"📚 Training on {len(subset)} samples (not just 5!)")
    else:
        subset = df.head(5)
        print(f"⚠️ Warning: Using only {len(subset)} samples - voice quality will suffer")
    
    model.train()
    best_loss = float('inf')
    
    print(f"\n🚀 Starting improved training for {epochs} epochs...")
    print(f"   - Learning rate: 2e-4 with cosine decay")
    print(f"   - Batch size: {batch_size}")
    print(f"   - Checkpoints saved every {save_every} epochs\n")
    
    for epoch in range(epochs):
        start_time = time.time()
        total_loss, count = 0, 0
        unique_sounds_total = 0
        
        # Shuffle data each epoch
        subset = subset.sample(frac=1, random_state=epoch).reset_index(drop=True)
        
        for idx, row in subset.iterrows():
            fname = str(row['filename']).strip()
            if not fname.endswith('.wav'): 
                fname += '.wav'
            path = os.path.join(WAV_DIR, fname)
            
            if not os.path.exists(path): 
                continue
            
            try:
                # Load & Resample audio
                wav, sr = torchaudio.load(path)
                if sr != 24000:
                    wav = torchaudio.transforms.Resample(sr, 24000)(wav)
                
                # Convert to mono if stereo
                if wav.shape[0] > 1:
                    wav = wav.mean(dim=0, keepdim=True)
                
                wav = wav.to(device)
                
                # Get ground truth audio tokens (ONLY first codebook)
                with torch.no_grad():
                    encoded = codec_model.encode(wav.unsqueeze(0))
                    # encoded[0][0] has shape [1, 8, T] - we only use first codebook
                    target = encoded[0][0][0, 0, :]  # Shape: [T]
                
                # Text to tokens
                phonemes = str(row['phonemes'])
                tokens = torch.tensor([char_to_id.get(c, 1) for c in phonemes]).unsqueeze(0).to(device)
                
                # Forward pass
                optimizer.zero_grad()
                logits = model(tokens)  # Shape: [1, seq_len, 1024]
                
                # FIXED: Use repeat_interleave instead of linear interpolation
                # This preserves discrete codebook structure
                text_len = logits.size(1)
                audio_len = target.size(0)
                
                if audio_len > text_len:
                    # Calculate repeat factor
                    repeat_factor = audio_len / text_len
                    
                    # Use repeat_interleave for proper alignment
                    repeat_sizes = []
                    for i in range(text_len):
                        start = int(i * repeat_factor)
                        end = int((i + 1) * repeat_factor)
                        repeat_sizes.append(max(1, end - start))
                    
                    # Adjust last element to match exact length
                    while sum(repeat_sizes) < audio_len:
                        repeat_sizes[-1] += 1
                    while sum(repeat_sizes) > audio_len:
                        repeat_sizes[-1] -= 1
                    
                    logits_expanded = []
                    for i, size in enumerate(repeat_sizes):
                        logits_expanded.extend([logits[0, i:i+1, :]] * size)
                    logits = torch.cat(logits_expanded, dim=1).unsqueeze(0)
                elif audio_len < text_len:
                    # Truncate if needed
                    logits = logits[:, :audio_len, :]
                
                # Calculate loss
                loss = criterion(logits.reshape(-1, 1024), target)
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                count += 1
                
                # Track unique sounds predicted
                pred_tokens = torch.argmax(logits[0], dim=-1)
                unique_sounds_total += len(torch.unique(pred_tokens))
                
            except Exception as e:
                print(f"  ⚠️ Error processing {fname}: {e}")
                continue
        
        # Update learning rate
        scheduler.step()
        
        # Logging
        if (epoch + 1) % 10 == 0 or epoch == 0:
            avg_loss = total_loss / max(count, 1)
            avg_unique = unique_sounds_total / max(count, 1)
            elapsed = time.time() - start_time
            current_lr = scheduler.get_last_lr()[0]
            
            print(f"Epoch {epoch+1:03d} | Loss: {avg_loss:.4f} | Unique Sounds: {avg_unique:.0f} | LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
        
        # Save checkpoints
        if (epoch + 1) % save_every == 0:
            checkpoint_path = os.path.join(CHECKPOINT_PATH, f'model_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': total_loss / max(count, 1),
                'vocab_size': VOCAB_SIZE,
                'char_to_id': char_to_id,
            }, checkpoint_path)
            print(f"  💾 Checkpoint saved: {checkpoint_path}")
            
            # Save best model
            if total_loss / max(count, 1) < best_loss:
                best_loss = total_loss / max(count, 1)
                best_path = os.path.join(CHECKPOINT_PATH, 'model_best.pth')
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_loss,
                    'vocab_size': VOCAB_SIZE,
                    'char_to_id': char_to_id,
                }, best_path)
                print(f"  🏆 New best model saved!")
    
    print(f"\n✅ Training complete! Best loss: {best_loss:.4f}")
    return model


# Run training with improved settings
# For Colab: 300-500 epochs recommended
train_fixed(epochs=300, batch_size=4, save_every=50, use_more_data=True)

In [ ]:
# ==========================================
# 6. AUDIO GENERATION (FIXED)
# ==========================================

def generate_nepali_fixed(index=0, temperature=0.9, repetition_penalty=1.1):
    """
    Generate speech with improved quality:
    - Proper codebook reconstruction
    - Temperature sampling for naturalness
    - Repetition penalty to avoid loops
    """
    model.eval()
    row = df.iloc[index]
    
    # Convert text to tokens
    phonemes = str(row['phonemes'])
    input_ids = torch.tensor([char_to_id.get(c, 1) for c in phonemes]).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_ids)  # [1, seq_len, 1024]
        
        # Apply temperature for more natural variation
        logits = logits / temperature
        
        # Sample instead of argmax for naturalness
        probs = F.softmax(logits, dim=-1)
        pred_tokens = torch.multinomial(probs.view(-1, 1024), 1).view(1, -1)
        
        # Build EnCodec format - ONLY first codebook, others zero
        # EnCodec expects [batch, 8, time]
        audio_len = pred_tokens.size(1)
        dummy_codes = torch.zeros((1, 8, audio_len), dtype=torch.long).to(device)
        dummy_codes[:, 0, :] = pred_tokens  # First codebook from model
        # Other 7 codebooks are zeros - EnCodec can still decode this!
        
        # Decode to audio
        wav_out = codec_model.decode([(dummy_codes, None)])
    
    print(f"📝 Text: {row['text']}")
    print(f"🔤 Phonemes: {phonemes}")
    print(f"🎵 Generated {audio_len} audio frames")
    
    return wav_out.cpu().squeeze().numpy()


# Load best model if available
best_model_path = os.path.join(CHECKPOINT_PATH, 'model_best.pth')
if os.path.exists(best_model_path):
    print(f"📥 Loading best model from {best_model_path}")
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded model from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
else:
    print("⚠️ No saved model found, using current model state")

# Test generation on multiple samples
print("\n" + "="*50)
print("🎤 TESTING AUDIO GENERATION")
print("="*50)

# Try first 3 samples
for i in range(min(3, len(df))):
    print(f"\n--- Sample {i+1} ---")
    try:
        audio = generate_nepali_fixed(index=i, temperature=0.8)
        display(ipd.Audio(audio, rate=24000))
    except Exception as e:
        print(f"❌ Error generating sample {i+1}: {e}")

In [ ]:
# ==========================================
# 7. CUSTOM TEXT GENERATION
# ==========================================

def generate_from_phonemes(phonemes_text, temperature=0.8):
    """Generate speech from custom phoneme string"""
    model.eval()
    
    # Convert phonemes to tokens
    input_ids = torch.tensor([char_to_id.get(c, 1) for c in phonemes_text]).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_ids)
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)
        pred_tokens = torch.multinomial(probs.view(-1, 1024), 1).view(1, -1)
        
        audio_len = pred_tokens.size(1)
        dummy_codes = torch.zeros((1, 8, audio_len), dtype=torch.long).to(device)
        dummy_codes[:, 0, :] = pred_tokens
        
        wav_out = codec_model.decode([(dummy_codes, None)])
    
    print(f"🔤 Input phonemes: {phonemes_text}")
    print(f"🎵 Generated {audio_len} audio frames")
    
    return wav_out.cpu().squeeze().numpy()


# Example: Generate custom phoneme sequence
# Replace with your own Nepali phonemes
custom_phonemes = "n ə m o s t e"  # Example: "namaste"
print(f"\n🎤 Generating custom phonemes: {custom_phonemes}")
audio = generate_from_phonemes(custom_phonemes, temperature=0.8)
display(ipd.Audio(audio, rate=24000))

## 💡 Tips for Better Results on Free Colab

1. **Train longer**: Aim for 500+ epochs for natural voice
2. **Use more data**: 100-200 samples minimum (you have 2064!)
3. **Save frequently**: Colab sessions timeout after ~12 hours
4. **Lower bandwidth**: 3.0 kbps trains faster than 6.0 kbps
5. **Monitor loss**: If loss plateaus, increase learning rate slightly
6. **Test often**: Generate audio every 50 epochs to track progress

## 🔧 If Still Getting Robotic Noise:

- Increase training samples to 300+
- Train for 1000+ epochs
- Try higher temperature (0.9-1.0) during generation
- Ensure your phoneme transcriptions are accurate
- Check audio quality of training samples